# Can we predict arrears next month?
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), 5-split walk-forward CV on training rows, selection by mean CV ROC AUC, holdout scored once.

**Feature decision:** `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. after/alongside the outcome (86% arrears when 1). It is not known at prediction time and is excluded as leakage. `meter_ref` and `billing_month` are excluded per the protocol.

In [ ]:
import json, hashlib, numpy as np, pandas as pd, yaml
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
proto = yaml.safe_load(open('protocol.yaml'))
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest() == proto['dataset']['sha256']
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest() == proto['split']['holdout_sha256']
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==proto['dataset']['rows'] and len(hold_idx)==proto['split']['holdout_rows']
is_hold = np.zeros(len(df), bool); is_hold[hold_idx] = True
train, hold = df[~is_hold], df[is_hold]
print(len(train), len(hold), train.billing_month.max(), hold.billing_month.min())

In [ ]:
TARGET='arrears_next_month'
NUM=['occupants','kwh','bill_eur','late_payments_12m']; CAT=['tariff']
FEATURES=NUM+CAT   # final_notice_sent excluded: recorded after the outcome
def pre(scale):
    return ColumnTransformer([('n', StandardScaler() if scale else 'passthrough', NUM),
                              ('c', OneHotEncoder(handle_unknown='ignore'), CAT)])
def mk(model, scale=False): return Pipeline([('pre',pre(scale)),('m',model)])
# all candidates named before any scoring
candidates = {
 'baseline_prior': mk(DummyClassifier(strategy='prior')),
 'logreg_C0.1': mk(LogisticRegression(C=0.1,max_iter=1000),True),
 'logreg_C1': mk(LogisticRegression(C=1,max_iter=1000),True),
 'rf_depth6': mk(RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1)),
 'hgb_lr0.05': mk(HistGradientBoostingClassifier(learning_rate=0.05,max_depth=3,max_iter=200,random_state=0)),
}
print(list(candidates))

In [ ]:
# walk-forward CV on training rows: months split into 6 consecutive blocks; fold k trains on blocks <=k, validates on block k+1
train = train.reset_index()  # 'index' = 0-based row position in data.csv
months = np.sort(train.billing_month.unique()); blocks = np.array_split(months, 6)
folds=[]
for k in range(5):
    tr = train.index[train.billing_month.isin(np.concatenate(blocks[:k+1]))].to_numpy()
    va = train.index[train.billing_month.isin(blocks[k+1])].to_numpy()
    folds.append((tr,va))
cv_scores={}
for name,m in candidates.items():
    cv_scores[name]=[roc_auc_score(train.loc[va,TARGET], clone(m).fit(train.loc[tr,FEATURES],train.loc[tr,TARGET]).predict_proba(train.loc[va,FEATURES])[:,1]) for tr,va in folds]
res=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T.sort_values('mean',ascending=False); res

In [ ]:
chosen = res['mean'].idxmax(); print('chosen:', chosen)
final = clone(candidates[chosen]).fit(train[FEATURES], train[TARGET])
p = final.predict_proba(hold[FEATURES])[:,1]
holdout_auc = roc_auc_score(hold[TARGET], p)   # scored once
print('holdout roc_auc', holdout_auc)

In [ ]:
ev = {'features':FEATURES, 'candidates':list(candidates),
 'cv_folds':[[[int(x) for x in train.loc[tr,'index']],[int(x) for x in train.loc[va,'index']]] for tr,va in folds],
 'cv_scores':{k:[float(x) for x in v] for k,v in cv_scores.items()},
 'chosen':chosen, 'holdout_rows':[int(x) for x in hold_idx],
 'holdout_scores':{chosen:float(holdout_auc)}}
json.dump(ev, open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. Without the leaking `final_notice_sent`, performance reflects what is genuinely predictable at billing time.